# PromptTemplate的使用

1、PromptTemplate如何获取实例 "重点"

2、两种特殊结构的使用（部分提示词模版、组合提示词）

3、给变量赋值的两种方式：format()/invoke()

4、结合大模型使用

## 1、PromptTemplate如何获取实例

方式一：使用构造方法的方式

In [3]:
from langchain_core.prompts import PromptTemplate

# 1、创建PromptTemplate实例
# 以下参数必须指明
prompt_template = PromptTemplate(
    template = "你是一个{role},你的名字是{name}。",
    input_variables = ["role","name"],
)
# print(prompt_template)
# 2、填充实例中变量
prompt = prompt_template.format(role = "人工智能专家",name = "小志")

print(prompt)

你是一个人工智能专家,你的名字是小志.


方式二：from_template() 推荐！！！

In [4]:
from langchain_core.prompts import PromptTemplate

# 1、创建PromptTemplate实例
prompt_template = PromptTemplate.from_template(template = "你是一个{role},你的名字是{name}。")
# print(prompt_template)
# 2、填充实例中变量
prompt = prompt_template.format(role = "人工智能专家",name = "小志")

print(prompt)

你是一个人工智能专家,你的名字是小志。


如果提示词模板中不包含变量，调用format()时不需要传入参数。

In [9]:
#1.导入相关的包
from langchain_core.prompts import PromptTemplate
# 2.定义提示词模版对象
text = """
Tell me a joke
"""
prompt_template = PromptTemplate.from_template(text)
# 3.默认使用f-string进行格式化（返回格式好的字符串）
prompt = prompt_template.format()
print(prompt)


Tell me a joke



## 2、两种特殊结构的使用（部分提示词模版、组合提示词）

### 2.1 部分提示词模版使用（重点）

①使用partial_variables()变量

In [12]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    template = "请评价{product}的优缺点,包括{aspect1}和{aspect2}。",
    partial_variables = {"aspect1":"电池续航"}
)

prompt = prompt_template.format(product = "手机",aspect2 = "价格")

print(prompt)

请评价手机的优缺点,包括电池续航和价格。


②调用方法partial()

In [14]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    template = "请评价{product}的优缺点,包括{aspect1}和{aspect2}。",
    # partial_variables = {"aspect1":"电池续航"}
)
# .partial()方法调用后原本的模版没变化，返回值的模版起作用
template = prompt_template.partial(aspect1="电池续航")
prompt = template.format(product = "手机",aspect2 = "价格")

print(prompt)

请评价手机的优缺点,包括电池续航和价格。


### 2.2 组合提示词使用

In [15]:
from langchain_core.prompts import PromptTemplate

template = (
        PromptTemplate.from_template("Tell me a joke about {topic}")
        + ", make it funny"
        + "\n\nand in {language}"
)
prompt = template.format(topic="sports", language="spanish")
print(prompt)

Tell me a joke about sports, make it funny

and in spanish


## 3、给变量赋值的两种方式：format()/invoke()

format():参数部分：给变量赋值，返回值：str类型

invoke():参数部分：字典，返回值：PromptValue

In [19]:
from langchain_core.prompts import PromptTemplate

# 1、创建PromptTemplate实例
prompt_template = PromptTemplate.from_template(template = "你是一个{role},你的名字是{name}。")

# 2、填充实例中变量
prompt = prompt_template.invoke(input = {"role":"人工智能专家","name":"小志"})

print(prompt)
print(type(prompt))

text='你是一个人工智能专家,你的名字是小志。'
<class 'langchain_core.prompt_values.StringPromptValue'>


## 4、结合大模型使用

In [22]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
import os
import dotenv

dotenv.load_dotenv()

os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
# 获取对话模型
chat_model = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature = 0.7,
    #max_tokens = 20,
)
# 生成提示词模版
template = PromptTemplate.from_template(template = "请论述{product}在{aspect}上的价值。")
# 模版变量赋值
prompt = template.invoke(input = {"product":"AI","aspect":"医疗"})

# 调用大模型
response = chat_model.invoke(prompt)

print(response.content)

人工智能（AI）在医疗领域的应用正迅速发展，展现出巨大的价值和潜力。以下是AI在医疗上的几个主要价值点：

1. **提高诊断准确性**：AI能够通过分析医疗图像（如X光片、CT扫描和MRI）来辅助医生进行更准确的诊断。例如，深度学习算法可以识别早期肿瘤或其他病变，从而提高早期发现的机会。

2. **个性化治疗**：AI可以分析患者的基因组数据、病史及其他相关信息，帮助医生制定个性化的治疗方案。这种精准医疗可以提高治疗效果，减少不必要的副作用。

3. **优化临床决策**：AI系统可以整合大量的医学文献和临床数据，为医生提供基于证据的决策支持。这不仅提高了医生的工作效率，也有助于提升患者的治疗效果。

4. **疾病预测与预防**：通过分析电子健康记录和其他数据，AI可以识别出高风险患者并预测疾病的发生。这为早期干预和预防提供了可能性，从而改善公共健康。

5. **自动化行政任务**：AI可以帮助医院和诊所自动化繁琐的行政流程，如病历管理、预约安排和保险理赔等，从而节省人力和时间，提高工作效率。

6. **药物研发**：AI在药物发现和研发中也发挥了重要作用。通过分析分子结构和生物数据，AI可以快速筛选出潜在的药物候选物，缩短研发周期并降低成本。

7. **远程医疗**：AI驱动的应用程序和设备可以帮助医生进行远程监测和诊断，尤其是在偏远地区或疫情期间。这种方式不仅提高了医疗服务的可及性，还能减轻医疗机构的压力。

8. **患者参与和教育**：AI可以通过聊天机器人和智能应用与患者互动，提供健康建议和教育信息，增强患者的自我管理能力和健康意识。

总结而言，AI在医疗领域的应用不仅能提升诊断和治疗的效率与准确性，还能改善患者体验和医疗服务质量。随着技术的不断进步，AI预计将在医疗行业发挥越来越重要的作用。
